# 12 — Learning to draw on the blank sheet (step 9b)

A DQN learns to draw on the 28 × 28 canvas of notebook 11 (81 actions, image target, 1-pixel tolerance).
Figures in the report style (`src/report_style.py`).

## The agent (`src/free_dqn.py`)

- **A convolutional network (CNN)** reads the observation images and outputs the 81 action values. Its first
  convolution keeps the full resolution: the lines are 1 pixel wide (a lighter network that halves the resolution
  at once covered only 61 % of L instead of 99 % after 20,000 steps).
- **Double DQN** (van Hasselt et al., 2016): the online network *chooses* the next action, the target network
  *evaluates* it:
  $$y = R + \gamma\, Q_{\theta^-}\big(o', \arg\max_a Q_\theta(o', a)\big) \qquad (y = R \text{ after STOP})$$
  which reduces the over-estimation caused by the max.
- **A compact replay memory**: images are stored as bits and rebuilt when a mini-batch is drawn.
- **Stabilised training**: with the settings of notebook 07 (learning rate 0.0005), learning on several letters
  rose and then collapsed. Here: learning rate 0.0001, target network every 5,000 steps, gradient norm clipped
  at 10 (each correction is bounded), memory of 200,000 transitions.

## Two ways of seeing the sheet

- **Whole sheet** (notebook 11): target, drawing, pen position, pen down — 4 images of 28 × 28.
- **Egocentric** ("the sheet seen from the pen"): a 41 × 41 window **centred on the pen**, with 4 channels:
  what is still **missing** of the letter, the **drawing**, the **inside of the sheet**, **pen down**.
  The pen is always at the centre, so "a piece is missing just to my right" always looks the same, wherever the
  pen is and whatever the letter. This is the image version of the local view that unlocked generalisation in
  step 7c. 41 × 41 is enough to see the whole letter area from any pen position on the letter.

| Parameter | Value |
|---|---|
| training letters | the 10 letters of the project (a random one at each episode, random natural start) |
| steps | 500,000 |
| ε | 1 → 0.05 over the first 30 % of the steps |
| γ, mini-batch, gradient step | 0.99, 64, every 4 steps |
| learning rate, target network, gradient clipping | 0.0001, every 5,000 steps, norm 10 |
| episode limit | 60 steps |

In [ ]:
import sys
sys.path.insert(0, "..")

import csv
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image

from src.letters import TRAINING_LETTERS, TEST_LETTERS
from src.free_draw import FreeDrawEnv
from src.free_dqn import ConvQNetwork, train_free_dqn, greedy_free_episode, EGO
from src import report_style as rs

rs.apply()
GAMMA = 0.99
RETRAIN = False          # True: train the 10-letter network again (1.5-3 h on a laptop CPU)
RETRAIN_TRIALS = False   # True: recompute the 2-letter comparison of section 1 (about 35 min)
STABLE = dict(lr=1e-4, target_update=5_000, buffer_size=200_000, grad_clip=10.0, double=True)

## 1. Whole sheet or sheet seen from the pen? (2 letters, L and T)

Exploratory runs (100,000 steps each) that decided the design. The numbers come from the training logs of these
runs (`results/12_trials_two_letters.csv`); set `RETRAIN_TRIALS = True` to recompute them.

In [ ]:
TRIALS = "../results/12_trials_two_letters.csv"
if RETRAIN_TRIALS:
    configs = {"whole sheet (lr 5e-4)": dict(total_steps=80_000, eps_fraction=0.4, grad_clip=None),
               "whole sheet (stabilised)": dict(total_steps=100_000, eps_fraction=0.4, **STABLE),
               "egocentric (stabilised)": dict(total_steps=100_000, eps_fraction=0.4, egocentric=True, **STABLE)}
    rows = []
    for label, kw in configs.items():
        res = train_free_dqn(["L", "T"], learning_starts=2_000, eval_every=20_000, seed=0, verbose=False, **kw)
        for step, r in zip(res["eval_steps"], res["eval_results"]):
            rows.append({"config": label, "steps": step,
                         "coverage": round(100 * np.mean([v["coverage"] for v in r.values()])),
                         "perfect": round(100 * np.mean([v["perfect"] for v in r.values()]))})
        print(label, "done")
    with open(TRIALS, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

trials = {}
with open(TRIALS) as f:
    for row in csv.DictReader(f):
        trials.setdefault(row["config"], []).append((int(row["steps"]) / 1000, float(row["coverage"]), float(row["perfect"])))

styles = {"whole sheet (lr 5e-4)": (rs.SLATE, ":"), "whole sheet (stabilised)": (rs.OCHRE, "--"),
          "egocentric (stabilised)": (rs.NAVY, "-")}
labels = {"whole sheet (lr 5e-4)": "whole sheet, learning rate 0.0005 (unstable)",
          "whole sheet (stabilised)": "whole sheet, stabilised", "egocentric (stabilised)": "sheet seen from the pen, stabilised"}
fig, axes = plt.subplots(1, 2, figsize=(rs.WIDTH, 2.5))
for config, pts in trials.items():
    color, ls = styles[config]
    x, cov, perf = np.array(pts).T
    axes[0].plot(x, cov, color=color, ls=ls, lw=1.5, marker="o", ms=3)
    axes[1].plot(x, perf, color=color, ls=ls, lw=1.5, marker="o", ms=3)
for ax, ylabel in zip(axes, ["coverage (%)", "perfect drawings (%)"]):
    ax.set_ylim(-5, 105)
    rs.axes_style(ax, "training steps (thousands)", ylabel)
fig.legend([plt.Line2D([], [], color=styles[c][0], ls=styles[c][1], lw=1.5, marker="o", ms=3) for c in styles],
           [labels[c] for c in styles], loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.09))
fig.tight_layout()
fig.savefig("../results/12_two_letters.png")
plt.show()

## 2. The egocentric agent on the 10 letters

The network trained by the script of step 9b is loaded from `results/12_free_dqn_10letters.pt` (with its learning
curves). If it is missing, or if `RETRAIN = True`, it is trained here (1.5–3 hours).

In [ ]:
MODEL, CURVES = "../results/12_free_dqn_10letters.pt", "../results/12_free_dqn_10letters_curves.npz"
metrics = ("coverage", "precision", "perfect", "stopped")
if RETRAIN or not (os.path.exists(MODEL) and os.path.exists(CURVES)):
    res = train_free_dqn(TRAINING_LETTERS, total_steps=500_000, eps_fraction=0.3, egocentric=True,
                         eval_every=25_000, seed=0, **STABLE)
    torch.save(res["net"].state_dict(), MODEL)
    np.savez(CURVES, eval_steps=res["eval_steps"],
             **{m: np.array([[r[n][m] for n in TRAINING_LETTERS] for r in res["eval_results"]]) for m in metrics},
             episode_returns=res["episode_returns"], time=res["time"])
net = ConvQNetwork(input_size=EGO)
net.load_state_dict(torch.load(MODEL))
net.eval()
curves = np.load(CURVES)
print(f"training time: {float(curves['time']) / 60:.0f} min")

fig, axes = plt.subplots(1, 2, figsize=(rs.WIDTH, 2.4))
steps = curves["eval_steps"] / 1000
axes[0].plot(steps, 100 * curves["coverage"].mean(1), color=rs.NAVY, lw=1.5, label="coverage")
axes[0].plot(steps, 100 * curves["precision"].mean(1), color=rs.TEAL, ls="--", lw=1.5, label="precision")
axes[1].plot(steps, 100 * curves["perfect"].mean(1), color=rs.NAVY, lw=1.5, label="perfect drawings")
axes[1].plot(steps, 100 * curves["stopped"].mean(1), color=rs.OCHRE, ls="--", lw=1.5, label="episodes ended by STOP")
for ax in axes:
    ax.set_ylim(-5, 105)
    rs.axes_style(ax, "training steps (thousands)", "% (mean over the 10 letters)")
    ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("../results/12_learning_curves.png")
plt.show()

In [ ]:
def play_all(names):
    out = {}
    for name in names:
        env = FreeDrawEnv(name)
        eps = [greedy_free_episode(net, env, s, GAMMA, egocentric=True) for s in env.start_pixels]
        out[name] = (env, eps)
    return out

train_eps = play_all(TRAINING_LETTERS)
print(f"{'letter':<7}{'coverage':>10}{'precision':>11}{'perfect':>9}{'STOP':>7}    (mean over the natural starting points)")
for name, (env, eps) in train_eps.items():
    print(f"{name:<7}{np.mean([e['coverage'] for e in eps]):>10.0%}{np.mean([e['precision'] for e in eps]):>11.0%}"
          f"{np.mean([e['perfect'] for e in eps]):>9.0%}{np.mean([e['stopped'] for e in eps]):>7.0%}")

fig, axes = rs.panel_grid(10, 5, panel=1.3)
for ax, (name, (env, eps)) in zip(axes, train_eps.items()):
    e = eps[0]
    rs.draw_canvas(ax, env.target, env.zone, e["history"], f"{name}: {e['coverage']:.0%} covered{'' if e['stopped'] else ', no STOP'}")
rs.add_caption_line(fig, rs.CANVAS_LEGEND)
fig.tight_layout(pad=0.4)
fig.savefig("../results/12_drawings_10_letters.png")
plt.show()

In [ ]:
def animate_canvas(items, path, ncols=5, fps=5):
    """GIF of several drawings played step by step. items = list of (env, history, title)."""
    nrows = int(np.ceil(len(items) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.4, nrows * 1.6))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[len(items):]:
        ax.axis("off")
    n = max(len(h) for _, h, _ in items) + 1

    def update(f):
        for ax, (env, h, title) in zip(axes, items):
            ax.clear()
            rs.draw_canvas(ax, env.target, env.zone, h[:min(f, len(h))], title)
        return []
    fig.tight_layout(pad=0.3)
    FuncAnimation(fig, update, frames=list(range(n)) + [n - 1] * 2 * fps).save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)

animate_canvas([(env, eps[0]["history"], name) for name, (env, eps) in train_eps.items()], "../results/12_drawings_10_letters.gif")
Image(filename="../results/12_drawings_10_letters.gif")

## 3. A first look at the 16 unseen letters

Without any new training: does the egocentric network, trained on 10 letters only, draw letters it has never
seen? (Notebook 08: the grid DQN trained on 10 letters drew 0 % of them; step 9c will train on random figures.)

In [ ]:
test_eps = play_all(TEST_LETTERS)
print(f"{'letter':<7}{'coverage':>10}{'precision':>11}{'perfect':>9}{'STOP':>7}")
for name, (env, eps) in test_eps.items():
    print(f"{name:<7}{np.mean([e['coverage'] for e in eps]):>10.0%}{np.mean([e['precision'] for e in eps]):>11.0%}"
          f"{np.mean([e['perfect'] for e in eps]):>9.0%}{np.mean([e['stopped'] for e in eps]):>7.0%}")
print(f"\nmean over the 16 unseen letters: perfect {np.mean([np.mean([e['perfect'] for e in eps]) for _, eps in test_eps.values()]):.0%}, "
      f"coverage {np.mean([np.mean([e['coverage'] for e in eps]) for _, eps in test_eps.values()]):.0%}")

fig, axes = rs.panel_grid(16, 8, panel=1.0)
for ax, (name, (env, eps)) in zip(axes, test_eps.items()):
    e = eps[0]
    rs.draw_canvas(ax, env.target, env.zone, e["history"], f"{name}: {e['coverage']:.0%}")
rs.add_caption_line(fig, rs.CANVAS_LEGEND)
fig.tight_layout(pad=0.3)
fig.savefig("../results/12_unseen_letters.png")
plt.show()

## What to notice

**1. Seeing the sheet from the pen is what makes learning possible** (section 1, letters L and T).
- Whole sheet, learning rate 0.0005: coverage rises to 80 % and then collapses — the instability of notebook 07,
  made worse by images and 81 actions.
- Whole sheet, stabilised: stable, coverage up to 86 %, but **no perfect drawing** in 100,000 steps (in the right
  panel the two whole-sheet curves stay at 0 and lie on top of each other).
- Sheet seen from the pen (egocentric), stabilised: **100 % perfect drawings** after 100,000 steps. Same network,
  same algorithm, same rewards: only the way of seeing changes. With the whole sheet, the network has to find the
  pen in the image and relate it to the letter by itself; centred on the pen, "a piece is missing just to my right"
  always looks the same.

**2. On the 10 letters: 100 % perfect drawings after 500,000 steps** (74 min on a laptop CPU).
- Every letter, from every natural starting point: 100 % of the skeleton covered, 100 % of the ink on the letter,
  ended by STOP. The agent finds by itself which move lengths and directions follow each stroke, including the
  diagonals of N, Z, V and X.
- The learning curves have three phases:
  - up to 50,000 steps, coverage jumps to 81 %: the +0.2 per new pixel is an immediate reward, easy to learn;
  - between 75,000 and 175,000 steps, coverage drops to about 50 % while ε goes down, before rising steadily;
  - perfect drawings only appear after 150,000 steps, and their curve almost coincides with the STOP curve: the
    agent stops only when the letter is complete, so **learning when to stop is the hard part** (its +2 comes at the
    very end: the delayed reward of notebook 06).
- Precision stays above 95 % from 50,000 steps on: drawing outside costs −0.2 per pixel, and the agent learns that very early.
- Honest reading: the curve is still rising at the end (88 % at 475,000 steps, 100 % at 500,000). More steps would
  make the result more robust; we keep the final network, the one saved in `results/`.

**3. A free style, not always the shortest one.** The drawings follow the letters exactly, but some pen-up moves
wander (E, H): a pen-up move costs only −0.05, so a detour of a few steps costs very little, and nothing pushes
towards the straight path.

**4. No generalisation yet: the 16 unseen letters fail** (section 3).
- 0 % perfect drawings, 21 % coverage on average. On most letters the pen does not even draw: the greedy agent
  repeats a move that would leave the sheet, the move is refused, the observation does not change, so it repeats
  it until the 60-step limit. On M it covers 98 % of the letter but goes back and forth on the same piece and
  never stops.
- Same conclusion as notebook 08 on the grid: trained on 10 letters only, the agent learns these 10 letters, not
  "drawing in general". Step 9c applies the remedy of notebook 09: train on **random figures** (the egocentric view
  is already the image version of the rich observation), then test on all 26 letters.
